# tau-controller demo

PID controller for τ — the publisher's relevance threshold.

The publisher sets one number: what percentage of conversations should include a recommendation. The controller adjusts τ to hit that target.

Run each cell to see it work.

In [ ]:
from pid import TauController, RecommendationGate
import random
from unittest.mock import patch

## 1. Basic usage

Set a target rate, start conversations, check if each should get a recommendation.

In [ ]:
controller = TauController(target_rate=0.10)  # 10% of conversations
gate = RecommendationGate(controller=controller, update_interval=50)

print(f"Initial τ: {controller.tau}")
print(f"Target rate: {controller.target_rate:.0%}")

In [ ]:
# Simulate a single conversation with 5 turns
gate.tracker.start("conv-1")

for turn in range(5):
    best_ad_distance = random.uniform(0, 3)  # random ad distance
    result = gate.should_recommend("conv-1", best_ad_distance)
    print(f"Turn {turn+1}: distance={best_ad_distance:.2f}, recommend={result}")

gate.on_conversation_end("conv-1")
print(f"\nτ after conversation: {controller.tau:.4f}")

## 2. Convergence

Run 3,000 conversations and watch τ converge to hit the target rate.

In [ ]:
import matplotlib.pyplot as plt

random.seed(42)
ctrl = TauController(target_rate=0.10, tau=1.5)
gate = RecommendationGate(controller=ctrl, update_interval=50)

tau_history = []
rate_chunks = []
chunk_recs = 0
chunk_total = 0

with patch("pid.time") as mock_time:
    t = 0.0
    ctrl._prev_time = t

    for i in range(3000):
        t += 0.01
        mock_time.monotonic.return_value = t

        cid = f"conv-{i}"
        gate.tracker.start(cid)
        best_distance = random.uniform(0.0, 3.0)

        recommended = False
        for _ in range(random.randint(1, 15)):
            if gate.should_recommend(cid, best_ad_distance=best_distance):
                recommended = True

        gate.on_conversation_end(cid)
        tau_history.append(ctrl.tau)

        chunk_total += 1
        if recommended:
            chunk_recs += 1
        if chunk_total == 50:
            rate_chunks.append(chunk_recs / chunk_total)
            chunk_recs = 0
            chunk_total = 0

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(tau_history, color="steelblue")
axes[0].set_ylabel("τ")
axes[0].set_title("Tau convergence (target = 10%)")
axes[0].grid(True, alpha=0.3)

axes[1].plot([i*50 for i in range(len(rate_chunks))], rate_chunks, color="steelblue")
axes[1].axhline(y=0.10, color="red", linestyle="--", alpha=0.5, label="Target")
axes[1].set_ylabel("Recommendation Rate")
axes[1].set_xlabel("Conversations")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final τ: {ctrl.tau:.3f}")
print(f"Last chunk rate: {rate_chunks[-1]:.0%}")

## 3. Try your own target

Change `my_target` and re-run to see how τ adjusts.

In [ ]:
my_target = 0.25  # <-- change this (0.01 to 0.50)

random.seed(42)
ctrl = TauController(target_rate=my_target, tau=1.5)
gate = RecommendationGate(controller=ctrl, update_interval=50)

tau_history = []
rate_chunks = []
chunk_recs = 0
chunk_total = 0

with patch("pid.time") as mock_time:
    t = 0.0
    ctrl._prev_time = t

    for i in range(5000):
        t += 0.01
        mock_time.monotonic.return_value = t

        cid = f"conv-{i}"
        gate.tracker.start(cid)
        best_distance = random.uniform(0.0, 3.0)

        recommended = False
        for _ in range(random.randint(1, 15)):
            if gate.should_recommend(cid, best_ad_distance=best_distance):
                recommended = True

        gate.on_conversation_end(cid)
        tau_history.append(ctrl.tau)

        chunk_total += 1
        if recommended:
            chunk_recs += 1
        if chunk_total == 100:
            rate_chunks.append(chunk_recs / chunk_total)
            chunk_recs = 0
            chunk_total = 0

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(tau_history, color="steelblue")
axes[0].set_ylabel("τ")
axes[0].set_title(f"Tau convergence (target = {my_target:.0%})")
axes[0].grid(True, alpha=0.3)

axes[1].plot([i*100 for i in range(len(rate_chunks))], rate_chunks, color="steelblue")
axes[1].axhline(y=my_target, color="red", linestyle="--", alpha=0.5, label="Target")
axes[1].set_ylabel("Recommendation Rate")
axes[1].set_xlabel("Conversations")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final τ: {ctrl.tau:.3f}")
print(f"Last chunk rate: {rate_chunks[-1]:.0%}")
print(f"Theoretical equilibrium τ: {my_target * 3.0:.3f}")

## 4. Shock recovery

A new advertiser category enters at conversation 1500. Ad distances suddenly halve. Watch τ tighten to maintain the target.

In [ ]:
random.seed(42)
ctrl = TauController(target_rate=0.10, tau=1.5)
gate = RecommendationGate(controller=ctrl, update_interval=50)

tau_history = []
rate_chunks = []
chunk_recs = 0
chunk_total = 0
shock_at = 1500

with patch("pid.time") as mock_time:
    t = 0.0
    ctrl._prev_time = t

    for i in range(3000):
        t += 0.01
        mock_time.monotonic.return_value = t

        cid = f"conv-{i}"
        gate.tracker.start(cid)

        ad_range = 1.5 if i >= shock_at else 3.0
        best_distance = random.uniform(0.0, ad_range)

        recommended = False
        for _ in range(random.randint(1, 15)):
            if gate.should_recommend(cid, best_ad_distance=best_distance):
                recommended = True

        gate.on_conversation_end(cid)
        tau_history.append(ctrl.tau)

        chunk_total += 1
        if recommended:
            chunk_recs += 1
        if chunk_total == 50:
            rate_chunks.append(chunk_recs / chunk_total)
            chunk_recs = 0
            chunk_total = 0

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(tau_history, color="steelblue")
axes[0].axvline(x=shock_at, color="red", linestyle="--", alpha=0.5, label="New category enters")
axes[0].set_ylabel("τ")
axes[0].set_title("Shock recovery")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot([i*50 for i in range(len(rate_chunks))], rate_chunks, color="steelblue")
axes[1].axhline(y=0.10, color="gray", linestyle="--", alpha=0.5, label="Target")
axes[1].axvline(x=shock_at, color="red", linestyle="--", alpha=0.5, label="Shock")
axes[1].set_ylabel("Recommendation Rate")
axes[1].set_xlabel("Conversations")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()